# FEATURE ENGINEERING

In [98]:
import pandas as pd
import numpy as np
import seaborn as sns
import os
import ast

In this file we will do the following:

1. Load the data + set index (sort)
2. Create the target variable
3. Feature engineering (create the features)
4. Visualization

### LOAD THE DATA AND SET INDEX

We are first going to create a final dataset with all the information. We start by loading all the clean datasets, and checking the information in them.

In [99]:
# ACLED
acled = pd.read_csv("../data_clean/acled_clean.csv")
acled = acled.set_index(['iso3', 'month']).sort_index() 

list_cols = [
    "event_type",
    "sub_event_type",
    "disorder_type",
]

for col in list_cols:
    acled[col] = acled[col].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

# IDMC
idmc = pd.read_csv("../data_clean/idmc_clean.csv")
idmc = idmc.set_index(['iso3', 'month']).sort_index()

# HDX
hdx = pd.read_csv("../data_clean/hdx_clean.csv")
hdx = hdx.set_index(['iso3', 'month']).sort_index()

hdx["hdx_alert_level"] = hdx["hdx_alert_level"].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )

# ECONAI
econAI = pd.read_csv("../data_clean/econAI_clean.csv")
econAI = econAI.set_index(['iso3', 'month']).sort_index()

# GOOGLE TRENDS
google = pd.read_csv("../data_clean/google_trends_clean.csv")
google = google.set_index(['iso3', 'month']).sort_index()

# INFORM INDEX
inform = pd.read_csv("../data_clean/inform_clean.csv")
inform = inform.set_index(['iso3', 'month']).sort_index()

In [100]:
datasets = {
    "IDMC": idmc,
    "ACLED": acled,
    "HDX": hdx,
    "EconAI": econAI,
    "Google Trends": google,
    "INFORM Index": inform
}

def analyze_datasets(datasets_dict):
    analysis = []
    
    for name, df in datasets_dict.items():
       
        temp_df = df.copy()
        temp_df = temp_df.reset_index()
        temp_df['month'] = pd.to_datetime(temp_df['month'])
        
        analysis.append({
            "Dataset": name,
            "Rows": len(temp_df),
            "Countries": temp_df['iso3'].nunique(),
            "Min_Date": temp_df['month'].min().strftime('%Y-%m'),
            "Max_Date": temp_df['month'].max().strftime('%Y-%m'),
            "Avg_Months": round(len(temp_df) / temp_df['iso3'].nunique(), 1)
        })
    
    return pd.DataFrame(analysis)

# Torna a executar-ho amb els teus dataframes
df_info = analyze_datasets(datasets)
print(df_info.to_string(index=False))

      Dataset  Rows  Countries Min_Date Max_Date  Avg_Months
         IDMC  8844         89  2018-01  2026-04        99.4
        ACLED 25236        238  1996-12  2026-03       106.0
          HDX  1209        107  1998-05  2026-03        11.3
       EconAI 35472        182  2010-01  2026-03       194.9
Google Trends  8888         88  2017-12  2026-04       101.0
 INFORM Index 25212        191  2016-05  2027-04       132.0


In [101]:
def check_temporal_gaps(df):
    df = df.copy().reset_index()
    df['month'] = pd.to_datetime(df['month'])
    df = df.sort_values(['iso3', 'month'])
    
    df['diff'] = df.groupby('iso3')['month'].diff() / pd.Timedelta(days=31)
    
    gaps = df[df['diff'] > 1.1]
    
    if gaps.empty:
        print("No temporal gaps found in the dataset.")
    else:
        print(f"{len(gaps)} temporal gaps found:")
        print(gaps[['iso3', 'month', 'diff']].head(10))
    return gaps

gaps_df = check_temporal_gaps(idmc)
gaps_df = check_temporal_gaps(acled)
gaps_df = check_temporal_gaps(hdx)
gaps_df = check_temporal_gaps(econAI)
gaps_df = check_temporal_gaps(google)
gaps_df = check_temporal_gaps(inform)

No temporal gaps found in the dataset.
2512 temporal gaps found:
   iso3      month      diff
1   ABW 2018-09-01  5.935484
2   ABW 2019-02-01  4.935484
4   ABW 2019-06-01  2.967742
8   ABW 2020-03-01  5.870968
9   ABW 2020-05-01  1.967742
11  ABW 2020-08-01  1.967742
12  ABW 2020-10-01  1.967742
14  ABW 2021-01-01  1.967742
15  ABW 2021-03-01  1.903226
16  ABW 2021-09-01  5.935484
965 temporal gaps found:
   iso3      month       diff
2   AFG 2018-02-01  10.870968
3   AFG 2018-05-01   2.870968
5   AFG 2018-09-01   2.967742
6   AFG 2019-03-01   5.838710
7   AFG 2019-09-01   5.935484
8   AFG 2020-04-01   6.870968
9   AFG 2020-08-01   3.935484
11  AFG 2021-04-01   6.838710
14  AFG 2021-10-01   3.935484
16  AFG 2022-06-01   6.838710
No temporal gaps found in the dataset.
No temporal gaps found in the dataset.
No temporal gaps found in the dataset.


Not a surprise that both ACLED and HDX Signals have temporal lags, as they only contain information for the date ans country where an event or alarm occur. So it's not a problem to have them.

So, regarning the missing data:

- ACLED: Missing values in the period covered by ACLED means that nothing happened there, so can be imputed (even a hole country).
- IDMC: Missing values for an entire country during the period covered by IDMC means that there hasen't been any displacement in that country, so we can impute them easily. We can't impute a gap but yes a hole country.
- HDX Signals: Same as ACLED.
- ECONAI: We can't impute anything.
- GOOGLE TRENDS: We can't impute anything
- INFORM Index: We can't impute anything

Based on that our dataset should contain only the countries contained in the intersection on ECONAI, GOOGLE trends and INFORM index. As Google trends we have scraped manually we just have the countries needed (the ones in thet intersection of EconAI and inform, during the periods covered by both of them), we don't have to worry about this one. Regarding the time periods, we need to keep only the periods covered by the intersection on IDMC, ECONAI, Google Trends and Inform index, that in this case is from 2018-01 to 2026-03.

In [102]:
# 1. Get the strict intersection of countries present in BOTH EconAI and INFORM
econAI_countries = set(econAI.reset_index()['iso3'].unique())
inform_countries = set(inform.index.get_level_values('iso3').unique())

# The target list: countries that MUST be in both datasets
target_countries = econAI_countries.intersection(inform_countries)

print("=" * 70)
print(f"🎯 TARGET COUNTRIES (Intersection of EconAI & INFORM): {len(target_countries)}")
print("=" * 70)

# 2. Check how many countries from each dataset are discarded/lost based on this target
for name, df_actual in datasets.items():
    actual_countries = set(df_actual.reset_index()['iso3'].unique())
    
    # Countries that are in the current dataset but WILL BE LOST 
    # because they are not in our target intersection
    discarded = actual_countries - target_countries
    
    # Countries that this dataset lacks to reach the target (if any)
    missing_from_target = target_countries - actual_countries
    
    print(f"📊 {name}:")
    print(f"   • Total countries in raw file: {len(actual_countries)}")
    print(f"   • Kept for analysis: {len(actual_countries.intersection(target_countries))}")
    
    if len(discarded) > 0:
        print(f"   ❌ Discarded countries (not in intersection): {len(discarded)} {sorted(list(discarded))}")
    else:
        print("   ✅ Perfect! No countries discarded from this dataset.")
        
    if len(missing_from_target) > 0:
        print(f"   ⚠️ Lacks these target countries (will cause NaNs): {sorted(list(missing_from_target))}")
        
    print("-" * 70)

🎯 TARGET COUNTRIES (Intersection of EconAI & INFORM): 176
📊 IDMC:
   • Total countries in raw file: 89
   • Kept for analysis: 86
   ❌ Discarded countries (not in intersection): 3 ['AB9', 'MYT', 'NCL']
   ⚠️ Lacks these target countries (will cause NaNs): ['ALB', 'ARE', 'ARG', 'AUT', 'BEL', 'BGR', 'BHS', 'BLZ', 'BRB', 'BRN', 'BTN', 'BWA', 'CAN', 'CHE', 'CHL', 'CHN', 'CRI', 'CUB', 'CZE', 'DEU', 'DNK', 'DOM', 'DZA', 'ERI', 'ESP', 'EST', 'FIN', 'FJI', 'GAB', 'GEO', 'GNB', 'GNQ', 'GRD', 'GTM', 'GUY', 'HRV', 'HUN', 'IRL', 'ISL', 'JAM', 'JOR', 'JPN', 'KOR', 'KWT', 'LAO', 'LSO', 'LTU', 'LUX', 'LVA', 'MAR', 'MDA', 'MDV', 'MKD', 'MLT', 'MNE', 'MNG', 'MRT', 'MUS', 'MYS', 'NAM', 'NOR', 'NPL', 'NZL', 'OMN', 'PAN', 'POL', 'PRK', 'PRT', 'PRY', 'RWA', 'SAU', 'SEN', 'SGP', 'SRB', 'STP', 'SVK', 'SVN', 'SWE', 'SWZ', 'SYC', 'TKM', 'TLS', 'TON', 'TTO', 'TUN', 'URY', 'UZB', 'VNM', 'VUT', 'WSM']
----------------------------------------------------------------------
📊 ACLED:
   • Total countries in raw file:

Let's do now the final dataset containing the countries in the intersecction of EconAI and INFORM, and the time periods in the intersection of EconAI, Inform and IDMC that is the same as the intersection of EconAI and INFORM if we first delete all the data from 2016-05 to 2017-12 of INFORM:


In [103]:
inform = inform.reset_index()
inform = inform[pd.to_datetime(inform["month"]) >= pd.Timestamp("2018-01-01")]
inform = inform.set_index(['iso3', 'month']).sort_index()

In [104]:
print(f"Min month: {inform.index.get_level_values('month').min()}")
print(f"Max month: {inform.index.get_level_values('month').max()}")
print(f"Number of countries: {inform.index.get_level_values('iso3').nunique()}")

Min month: 2018-01-01
Max month: 2027-04-01
Number of countries: 191


In [105]:
df = econAI.join(inform, how="inner")

df = df.join(acled, how="left") \
        .join(hdx, how="left") \
        .join(idmc, how="left") \
        .join(google, how="left")

In [106]:
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.999608  1.000000       8.331437        9.280007     7.8   
     2018-02-01  0.996514  1.000000       8.295601        9.393767     7.8   
     2018-03-01  0.998218  1.000000       8.339758        9.346226     7.8   
     2018-04-01  0.995082  0.999815       8.383570        9.414723     7.8   
     2018-05-01  0.991237  0.998947       8.342653        9.481498     7.7   

                  VU   CC   HA  \
iso3 month                       
AFG  2018-01-01  7.1  7.7  8.8   
     2018-02-01  7.1  7.7  8.8   
     2018-03-01  7.1  7.7  8.8   
     2018-04-01  7.1  7.7  8.8   
     2018-05-01  7.1  7.5  8.7   

                                                        event_type  \
iso3 month                                                           
AFG  2018-01-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-02-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-03-01  [Battles, Explosions/Remote violence, Violence...   
     2018-04-01  [Battles, Explosions/Remote violence, Protests...   
     2018-05-01  [Battles, Explosions/Remote violence, Strategi...   

                                                    sub_event_type  ...  \
iso3 month                                                          ...   
AFG  2018-01-01  [Armed clash, Government regains territory, Ai...  ...   
     2018-02-01  [Armed clash, Air/drone strike, Grenade, Remot...  ...   
     2018-03-01  [Armed clash, Air/drone strike, Remote explosi...  ...   
     2018-04-01  [Armed clash, Non-state actor overtakes territ...  ...   
     2018-05-01  [Armed clash, Non-state actor overtakes territ...  ...   

                hdx_value  monthly_displacement  Flight Airport  Travel  \
iso3 month                                                                
AFG  2018-01-01       NaN           7456.583577     4.0     2.0     2.0   
     2018-02-01       1.0          13918.956010     3.0     2.0     2.0   
     2018-03-01       NaN          15410.272726     4.0     2.0     2.0   
     2018-04-01       NaN          17198.881440     3.0     2.0     2.0   
     2018-05-01       2.0          39769.719735     4.0     2.0     2.0   

                 Train  Bus  Passport  Travel visa  Right of asylum  
iso3 month                                                           
AFG  2018-01-01    1.0  1.0       1.0          4.0              0.0  
     2018-02-01    1.0  1.0       1.0          4.0              0.0  
     2018-03-01    0.0  1.0       1.0          4.0              0.0  
     2018-04-01    1.0  1.0       1.0          5.0              0.0  
     2018-05-01    1.0  1.0       1.0          5.0              0.0  

[5 rows x 24 columns]

In [107]:
df.size, df.shape

(418176, (17424, 24))

In [108]:
print(f"Min month: {df.index.get_level_values('month').min()}")
print(f"Max month: {df.index.get_level_values('month').max()}")
print(f"Number of countries: {df.index.get_level_values('iso3').nunique()}")

Min month: 2018-01-01
Max month: 2026-03-01
Number of countries: 176


Let's now solve the problem of the missing values:

- The missing values in `disorder_type`, `sub_event_type`, and `event_type` correspond to month x country that hasen't had any event. Then we will put a empty list.

- Missing values in `events` and `fatalities` correspond to months x countries with no events nor fatalities, so we will put 0 on both. 

- Missing values in `monthly_displacement` correspond to countries that hasn't had any displacement in all the covered period, so we are putting a 0.

- Missing values in `hdx_alert_level` correspond to the month x country with no hdx alert, so we put an empty list. Te same for `hdx_value`, so we are putting a 0.

In [109]:
df["event_type"] = df["event_type"].apply(
    lambda x: [] if x is None or (isinstance(x, float) and pd.isna(x)) else x
)
df["sub_event_type"] = df["sub_event_type"].apply(
    lambda x: [] if x is None or (isinstance(x, float) and pd.isna(x)) else x
)
df["disorder_type"] = df["disorder_type"].apply(
    lambda x: [] if x is None or (isinstance(x, float) and pd.isna(x)) else x
)
df["events"] = df["events"].fillna(0)
df["fatalities"] = df["fatalities"].fillna(0)
df["hdx_alert_level"] = df["hdx_alert_level"].apply(
    lambda x: [] if x is None or (isinstance(x, float) and pd.isna(x)) else x
)
df["hdx_value"] = df["hdx_value"].fillna(0)
df["monthly_displacement"] = df["monthly_displacement"].fillna(0)

In [110]:
df = df.sort_index()
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.999608  1.000000       8.331437        9.280007     7.8   
     2018-02-01  0.996514  1.000000       8.295601        9.393767     7.8   
     2018-03-01  0.998218  1.000000       8.339758        9.346226     7.8   
     2018-04-01  0.995082  0.999815       8.383570        9.414723     7.8   
     2018-05-01  0.991237  0.998947       8.342653        9.481498     7.7   

                  VU   CC   HA  \
iso3 month                       
AFG  2018-01-01  7.1  7.7  8.8   
     2018-02-01  7.1  7.7  8.8   
     2018-03-01  7.1  7.7  8.8   
     2018-04-01  7.1  7.7  8.8   
     2018-05-01  7.1  7.5  8.7   

                                                        event_type  \
iso3 month                                                           
AFG  2018-01-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-02-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-03-01  [Battles, Explosions/Remote violence, Violence...   
     2018-04-01  [Battles, Explosions/Remote violence, Protests...   
     2018-05-01  [Battles, Explosions/Remote violence, Strategi...   

                                                    sub_event_type  ...  \
iso3 month                                                          ...   
AFG  2018-01-01  [Armed clash, Government regains territory, Ai...  ...   
     2018-02-01  [Armed clash, Air/drone strike, Grenade, Remot...  ...   
     2018-03-01  [Armed clash, Air/drone strike, Remote explosi...  ...   
     2018-04-01  [Armed clash, Non-state actor overtakes territ...  ...   
     2018-05-01  [Armed clash, Non-state actor overtakes territ...  ...   

                hdx_value  monthly_displacement  Flight Airport  Travel  \
iso3 month                                                                
AFG  2018-01-01       0.0           7456.583577     4.0     2.0     2.0   
     2018-02-01       1.0          13918.956010     3.0     2.0     2.0   
     2018-03-01       0.0          15410.272726     4.0     2.0     2.0   
     2018-04-01       0.0          17198.881440     3.0     2.0     2.0   
     2018-05-01       2.0          39769.719735     4.0     2.0     2.0   

                 Train  Bus  Passport  Travel visa  Right of asylum  
iso3 month                                                           
AFG  2018-01-01    1.0  1.0       1.0          4.0              0.0  
     2018-02-01    1.0  1.0       1.0          4.0              0.0  
     2018-03-01    0.0  1.0       1.0          4.0              0.0  
     2018-04-01    1.0  1.0       1.0          5.0              0.0  
     2018-05-01    1.0  1.0       1.0          5.0              0.0  

[5 rows x 24 columns]

### CREATE THE TARGET VARIABLE

Our target variable is going to be an incidence variable showing if there's going to be a situation for which the country is elegible for an allocation in the next 2 months. To do so, we first need to create the variable allocation-elegible.

We say that a country is elegible to recieve an allocation if the country satisfies the necessary conditions for the CERF to send an allocation: 50,000 new internal displacements over a 3 months period.

Our target variable predicts whether in the following two months there's going to be a situation for whitch the CERF would send allocation. Eventhough we are interseted in predicting the first month for which this state is met, we don't want to put a 0 in the target variable, if in the next two months the condition is met but not for the first time, because that will confuse a lot our model. The other option was to put Nan in thoose cases, but considering that we have a super balanced datset, we choose to put 1.

In [111]:
df = df.reset_index()
df = df.sort_values(by=['iso3', 'month'])


df['rolling_3m_displacements'] = (
    df.groupby('iso3')['monthly_displacement']
    .rolling(window=3, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

df['allocation-eligible'] = (df['rolling_3m_displacements'] >= 50000).astype(int)
df = df.set_index(['iso3', 'month']).sort_index()

Now we can create our target variable:

In [112]:
# Create the target variable: 1 if conflict in t+1 or t+2, else 0
df["target_2m"] = (
    (df.groupby(level="iso3")["allocation-eligible"].shift(-1) == 1) |
    (df.groupby(level="iso3")["allocation-eligible"].shift(-2) == 1)
).astype(int)

print("Total positive targets (conflict in next 2 months):", df["target_2m"].sum())

Total positive targets (conflict in next 2 months): 777


### FEATURE ENGINEERING

#### CERF SIGNALS

We know from the CERF that depending on the state of a country in a determinate moment, it's susceptible of having one type of conflict or another (hard onset or protracted). The important thing is that depending on the state, the early signals that indicates an incoming crisis are different. So the first thing that we have to do is classify each county x month to one of the two states, and then, depending on it, check which signals are beeing seen. 

In order to classify a country x month in a state, we use the CERF logic, saying:

"A conflict is classified as a “protracted crisis” when EconAI’s risk score remains consistently above 0.6 over a period of 12 months."

Then, the early signals depending on the state are:

1. Protracted: 
- EconAI’s risk score increases by ≥0.1 over 3 consecutive months while already >0.7; OR
- Monthly displacement or fatalities increase by ≥1.5 times rolling 6-month average for 2 consecutive months; OR
- There are ≥3 Medium or High HDX signals occur within 90 days.

2. Hard onset:
- EconAI’s risk score increases by ≥0.3 within 2 months (this will typically also be confirmed by the issuance of an HDX signal the month of the significant increase); OR
- Monthly displacement increases by > 3 times the rolling 6-month average (this will typically be confirmed by a sharp increase in risk score). 

In instances where displacement data is missing, such as Armenia, the fatalities trend can be used instead.


In [113]:
# 1. CLASSIFY THE STATE (Protracted vs Hard Onset)

# Rule: Risk > 0.6 for 12 consecutive months
df['risk_gt_06'] = df['risk_3'] > 0.6

# We use rolling sum of 12 on the boolean column. If sum is 12, it was True for 12 consecutive months.
df['is_protracted'] = df.groupby(level='iso3')['risk_gt_06'].transform(
    lambda x: x.rolling(window=12, min_periods=12).sum() == 12
).astype(int)

df['state'] = np.where(df['is_protracted']==1, 'Protracted', 'Hard onset')

In [114]:
# 2. PRE-CALCULATE BASE METRICS

# 6-month rolling averages
df['disp_6m_avg'] = df.groupby(level='iso3')['monthly_displacement'].transform(lambda x: x.rolling(6).mean())
df['fat_6m_avg'] = df.groupby(level='iso3')['fatalities'].transform(lambda x: x.rolling(6).mean())

# Function to safely parse the HDX alert level list and count 'Medium' or 'High'
def count_hdx_med_high(x):
    if isinstance(x, list):
        return sum(1 for item in x if item in ['Medium concern', 'High concern'])
    elif isinstance(x, str): # In case pandas read the list as a string
        return 1 if 'Medium concern' in x or 'High concern' in x else 0
    return 0

df['hdx_med_high_count'] = df['hdx_alert_level'].apply(count_hdx_med_high)
# Rolling 90 days (3 months) sum of HDX alerts
df['hdx_3m_sum'] = df.groupby(level='iso3')['hdx_med_high_count'].transform(lambda x: x.rolling(3).sum())

In [115]:
# 3. PROTRACTED SIGNALS EVALUATION

# P1: Risk increases by >= 0.1 over 3 months while already > 0.7
df['p_sig1'] = (df.groupby(level='iso3')['risk_3'].diff(3) >= 0.1) & (df['risk_3'] > 0.7)

# P2: Displacement or fatalities >= 1.5x their 6-month average for 2 consecutive months
spike_1_5x = (df['monthly_displacement'] >= 1.5 * df['disp_6m_avg']) | (df['fatalities'] >= 1.5 * df['fat_6m_avg'])

# Agrupamos la Serie directamente usando su propio índice (level='iso3')
df['p_sig2'] = spike_1_5x.groupby(level='iso3').transform(
    lambda x: x.rolling(2).sum() == 2
)
# P3: >= 3 Medium or High HDX signals within 90 days
df['p_sig3'] = df['hdx_3m_sum'] >= 3

# Combine Protracted signals (True if any is True)
df['protracted_signal'] = df['p_sig1'] | df['p_sig2'] | df['p_sig3']


In [116]:
# 4. HARD ONSET SIGNALS EVALUATION

# H1: Risk increases by >= 0.3 within 2 months
df['h_sig1'] = df.groupby(level='iso3')['risk_3'].diff(2) >= 0.3

# H2: Displacement > 3x 6-month avg. If missing, use fatalities > 3x 6-month avg.
disp_spike_3x = df['monthly_displacement'] > 3 * df['disp_6m_avg']
fat_spike_3x = df['fatalities'] > 3 * df['fat_6m_avg']

# np.where lets us use the fatality logic ONLY when displacement is NaN
df['h_sig2'] = np.where(df['monthly_displacement'].isna(), fat_spike_3x, disp_spike_3x)

# Combine Hard Onset signals
df['hard_onset_signal'] = df['h_sig1'] | df['h_sig2']


In [117]:
# 5. FINAL EARLY SIGNAL COLUMN

# Apply the corresponding signal based on the state of the country that month
df['early_signal'] = np.where(
    df['state'] == 'Protracted',
    df['protracted_signal'],
    df['hard_onset_signal']
).astype(int) # Convert True/False to 1/0

# Let's check the distribution!
print(df['state'].value_counts())
print(f"\nTotal early signals detected: {df['early_signal'].sum()}")

df = df.drop(columns=['state'])

state
Hard onset    15716
Protracted     1708
Name: count, dtype: int64

Total early signals detected: 798


In [118]:
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.999608  1.000000       8.331437        9.280007     7.8   
     2018-02-01  0.996514  1.000000       8.295601        9.393767     7.8   
     2018-03-01  0.998218  1.000000       8.339758        9.346226     7.8   
     2018-04-01  0.995082  0.999815       8.383570        9.414723     7.8   
     2018-05-01  0.991237  0.998947       8.342653        9.481498     7.7   

                  VU   CC   HA  \
iso3 month                       
AFG  2018-01-01  7.1  7.7  8.8   
     2018-02-01  7.1  7.7  8.8   
     2018-03-01  7.1  7.7  8.8   
     2018-04-01  7.1  7.7  8.8   
     2018-05-01  7.1  7.5  8.7   

                                                        event_type  \
iso3 month                                                           
AFG  2018-01-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-02-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-03-01  [Battles, Explosions/Remote violence, Violence...   
     2018-04-01  [Battles, Explosions/Remote violence, Protests...   
     2018-05-01  [Battles, Explosions/Remote violence, Strategi...   

                                                    sub_event_type  ...  \
iso3 month                                                          ...   
AFG  2018-01-01  [Armed clash, Government regains territory, Ai...  ...   
     2018-02-01  [Armed clash, Air/drone strike, Grenade, Remot...  ...   
     2018-03-01  [Armed clash, Air/drone strike, Remote explosi...  ...   
     2018-04-01  [Armed clash, Non-state actor overtakes territ...  ...   
     2018-05-01  [Armed clash, Non-state actor overtakes territ...  ...   

                hdx_med_high_count  hdx_3m_sum  p_sig1 p_sig2  p_sig3  \
iso3 month                                                              
AFG  2018-01-01                  0         NaN   False  False   False   
     2018-02-01                  1         NaN   False  False   False   
     2018-03-01                  0         1.0   False  False   False   
     2018-04-01                  0         1.0   False  False   False   
     2018-05-01                  1         1.0   False  False   False   

                 protracted_signal  h_sig1  h_sig2  hard_onset_signal  \
iso3 month                                                              
AFG  2018-01-01              False   False   False              False   
     2018-02-01              False   False   False              False   
     2018-03-01              False   False   False              False   
     2018-04-01              False   False   False              False   
     2018-05-01              False   False   False              False   

                 early_signal  
iso3 month                     
AFG  2018-01-01             0  
     2018-02-01             0  
     2018-03-01             0  
     2018-04-01             0  
     2018-05-01             0  

[5 rows x 41 columns]

In [119]:
display(pd.DataFrame(df.columns, columns=['Column Name']))

,Column Name
0,risk_3
1,risk_12
2,logfat_risk_3
3,logfat_risk_12
4,INFORM
5,VU
6,CC
7,HA
8,event_type
9,sub_event_type


All the variables since here comming from ACLED, IDMC, ECONAI and HDX Signals, is what we will consider our Baseline/ CERF Signals, because are all the variables obtained from using the datasets provided by CERF, and applying the transformations to obtain the signals recommended by them.

#### HISTORICAL FEATURES

Now we are going to generate some derived features from all the datasets, capturing the historical state of the country. We will create some lags of some relevant variables, but also we are going to handle the categorical variables to make them useful for our model.

In [120]:
# Columns for which we want to obtain the derived features
cols_to_lag = ['monthly_displacement', 'fatalities', 'events', 'risk_3', 'risk_12', 'logfat_risk_3', 'logfat_risk_12']

for col in cols_to_lag:
    # Create the Lag 1 (previous month)
    df[f'{col}_lag1'] = df.groupby('iso3')[col].shift(1)
    
    # Create the Lag 2 (two months ago) only for displacement
    if col == 'monthly_displacement':
        df[f'{col}_lag2'] = df.groupby('iso3')[col].shift(2)

In [121]:
import ast

# Function to safely parse list-like strings and handle NaNs
def safe_parse_list(val):
    # If it's a list (because we've already processed the data), do nothing
    if isinstance(val, list):
        return val
        
    # If it's a string, we check if it has list format.
    if isinstance(val, str):
        val = val.strip()
        if val.startswith('[') and val.endswith(']'):
            try:
                parsed = ast.literal_eval(val)
                return parsed if isinstance(parsed, list) else [parsed]
            except (ValueError, SyntaxError):
                return []
        else:
            # If it's a normal string ("Attack"), we put it inside a list ["Attack"]
            return [val] if val != "" else []
            
    # If it's a NaN or an empty value, return an empty list to avoid errors
    return []

Regarding the column `hdx_alert_level`, we are going to obtain 3 features after doing ordinal encoding of the level of the signals. We are going to geet the maximum value for country x month, the mean value, and the sum.

In [122]:
# ORDINAL ENCODING for HDX Signals because they have a clear severity order (None < Medium concern < High concern)
hdx_mapping = {
    'None': 0,
    'Medium concern': 1,
    'High concern': 2
}

def get_max_hdx(val_list):
    scores = [hdx_mapping.get(str(x), 0) for x in val_list]
    return max(scores) if scores else 0

def get_sum_hdx(val_list):
    scores = [hdx_mapping.get(str(x), 0) for x in val_list]
    return sum(scores) if scores else 0

def get_mean_hdx(val_list):
    scores = [hdx_mapping.get(str(x), 0) for x in val_list]
    return np.mean(scores) if scores else 0.0

# We will create three different features: max, sum, and mean of the HDX alert levels for each month

df['hdx_alert_max'] = df['hdx_alert_level'].apply(safe_parse_list).apply(get_max_hdx)
df['hdx_alert_sum'] = df['hdx_alert_level'].apply(safe_parse_list).apply(get_sum_hdx)
df['hdx_alert_mean'] = df['hdx_alert_level'].apply(safe_parse_list).apply(get_mean_hdx)

Now, the three remaining categorical columns are the ones coming from acled: `event_type`, `sub_event_type`and `disorder_type`. As there are a lot of types of each one, we are not going to do one.hot encoding and create dummies, we are going to create useful variables from them.

In [123]:
def extract_acled_features(row):
    # 1. Safely parse the three raw columns using your safe_parse_list function
    events = safe_parse_list(row['event_type'])
    sub_events = safe_parse_list(row['sub_event_type'])
    disorders = safe_parse_list(row['disorder_type'])
    
    # 2. Consolidate everything into a single set of lowercase tags.
    # This also splits mixed categories like "Political violence; Demonstrations"
    all_tags = set()
    for item in (events + sub_events + disorders):
        if isinstance(item, str):
            if ';' in item:
                all_tags.update([x.strip().lower() for x in item.split(';')])
            else:
                all_tags.add(item.strip().lower())

    # 3. Strict classification based EXACTLY on your frequency lists
    
    # CATEGORY A: Direct violence targeting civilians and defenseless people
    is_civilian_targeted = 1 if any(x in all_tags for x in [
        'violence against civilians', 'attack', 'mob violence', 
        'abduction/forced disappearance', 'looting/property destruction', 
        'sexual violence', 'excessive force against protesters'
    ]) else 0
    
    # CATEGORY B: Armed combat, military warfare, and explosives (High Intensity)
    is_heavy_warfare = 1 if any(x in all_tags for x in [
        'battles', 'explosions/remote violence', 'armed clash', 
        'remote explosive/landmine/ied', 'air/drone strike', 
        'shelling/artillery/missile attack', 'grenade', 
        'suicide bomb', 'chemical weapon'
    ]) else 0
    
    # CATEGORY C: Protests, riots, and public mobilization (Social Unrest)
    is_social_unrest = 1 if any(x in all_tags for x in [
        'protests', 'riots', 'demonstrations', 'peaceful protest', 
        'violent demonstration', 'protest with intervention', 'arrests'
    ]) else 0
    
    # CATEGORY D: Changes in territorial control or base establishments
    is_territorial_shift = 1 if any(x in all_tags for x in [
        'government regains territory', 'non-state actor overtakes territory', 
        'non-violent transfer of territory', 'headquarters or base established'
    ]) else 0
    
    # CATEGORY E: Strategic developments, agreements, or covert operations
    is_strategic_or_other = 1 if any(x in all_tags for x in [
        'strategic developments', 'change to group/activity', 'other', 
        'disrupted weapons use', 'agreement'
    ]) else 0

    # SPECIAL BACKGROUND FLAG: Pure presence of structural political violence
    is_political_violence = 1 if 'political violence' in all_tags else 0

    # 4. Return the 6 new binary indicators as a Pandas Series
    return pd.Series({
        'acled_civilian_targeted': is_civilian_targeted,
        'acled_heavy_warfare': is_heavy_warfare,
        'acled_social_unrest': is_social_unrest,
        'acled_territorial_shift': is_territorial_shift,
        'acled_strategic_or_other': is_strategic_or_other,
        'acled_political_violence': is_political_violence
    })

In [124]:
acled_features = df.apply(extract_acled_features, axis=1)

# Join the new features back into your main final DataFrame
df = df.join(acled_features)

In [125]:
# Muestra las columnas en una tabla vertical interactiva
display(pd.DataFrame(df.columns, columns=['Column Name']))

,Column Name
0,risk_3
1,risk_12
2,logfat_risk_3
3,logfat_risk_12
4,INFORM
5,VU
6,CC
7,HA
8,event_type
9,sub_event_type


In [126]:
df.columns = df.columns.str.replace(" ", "_")

In [127]:
# Muestra las columnas en una tabla vertical interactiva
display(pd.DataFrame(df.columns, columns=['Column Name']))

,Column Name
0,risk_3
1,risk_12
2,logfat_risk_3
3,logfat_risk_12
4,INFORM
5,VU
6,CC
7,HA
8,event_type
9,sub_event_type


#### FEATURES FROM GOOGLE TRENDS

From google trends we have 8 different features corresponding to a multiple of the number of searches of certain words in google. We think that is very important to obtain derived features like lags, growth and rolling means, to see the evolution of the searches.

In [128]:
features_to_lag = ['Flight', 'Airport', 'Travel', 'Train', 'Bus', 'Passport', 'Travel_visa', 'Right_of_asylum']

# Create lags of 1, 2, and 3 months for all the features coming from Google Trends
for lag in [1, 2, 3]:
    for col in features_to_lag:
        df[f'{col}_lag_{lag}m'] = df.groupby('iso3')[col].shift(lag)


# Growth ratio compared to the previous month (Acceleration)
for col in features_to_lag:
    # Afegim un petit epsilon (+ 1e-5) per evitar la divisió per zero si la cerca era 0
    df[f'{col}_growth_1m'] = (df[col] - df.groupby('iso3')[col].shift(1)) / (df.groupby('iso3')[col].shift(1) + 1e-5)


# Rolling mean of the last 3 months
for col in features_to_lag:
    df[f'{col}_rolling_mean_3m'] = df.groupby('iso3')[col].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

In [129]:
df.head()

risk_3   risk_12  logfat_risk_3  logfat_risk_12  INFORM  \
iso3 month                                                                   
AFG  2018-01-01  0.999608  1.000000       8.331437        9.280007     7.8   
     2018-02-01  0.996514  1.000000       8.295601        9.393767     7.8   
     2018-03-01  0.998218  1.000000       8.339758        9.346226     7.8   
     2018-04-01  0.995082  0.999815       8.383570        9.414723     7.8   
     2018-05-01  0.991237  0.998947       8.342653        9.481498     7.7   

                  VU   CC   HA  \
iso3 month                       
AFG  2018-01-01  7.1  7.7  8.8   
     2018-02-01  7.1  7.7  8.8   
     2018-03-01  7.1  7.7  8.8   
     2018-04-01  7.1  7.7  8.8   
     2018-05-01  7.1  7.5  8.7   

                                                        event_type  \
iso3 month                                                           
AFG  2018-01-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-02-01  [Battles, Explosions/Remote violence, Strategi...   
     2018-03-01  [Battles, Explosions/Remote violence, Violence...   
     2018-04-01  [Battles, Explosions/Remote violence, Protests...   
     2018-05-01  [Battles, Explosions/Remote violence, Strategi...   

                                                    sub_event_type  ...  \
iso3 month                                                          ...   
AFG  2018-01-01  [Armed clash, Government regains territory, Ai...  ...   
     2018-02-01  [Armed clash, Air/drone strike, Grenade, Remot...  ...   
     2018-03-01  [Armed clash, Air/drone strike, Remote explosi...  ...   
     2018-04-01  [Armed clash, Non-state actor overtakes territ...  ...   
     2018-05-01  [Armed clash, Non-state actor overtakes territ...  ...   

                Travel_visa_growth_1m  Right_of_asylum_growth_1m  \
iso3 month                                                         
AFG  2018-01-01                   NaN                        NaN   
     2018-02-01              0.000000                        0.0   
     2018-03-01              0.000000                        0.0   
     2018-04-01              0.249999                        0.0   
     2018-05-01              0.000000                        0.0   

                 Flight_rolling_mean_3m Airport_rolling_mean_3m  \
iso3 month                                                        
AFG  2018-01-01                4.000000                     2.0   
     2018-02-01                3.500000                     2.0   
     2018-03-01                3.666667                     2.0   
     2018-04-01                3.333333                     2.0   
     2018-05-01                3.666667                     2.0   

                 Travel_rolling_mean_3m  Train_rolling_mean_3m  \
iso3 month                                                       
AFG  2018-01-01                     2.0               1.000000   
     2018-02-01                     2.0               1.000000   
     2018-03-01                     2.0               0.666667   
     2018-04-01                     2.0               0.666667   
     2018-05-01                     2.0               0.666667   

                 Bus_rolling_mean_3m  Passport_rolling_mean_3m  \
iso3 month                                                       
AFG  2018-01-01                  1.0                       1.0   
     2018-02-01                  1.0                       1.0   
     2018-03-01                  1.0                       1.0   
     2018-04-01                  1.0                       1.0   
     2018-05-01                  1.0                       1.0   

                 Travel_visa_rolling_mean_3m  Right_of_asylum_rolling_mean_3m  
iso3 month                                                                     
AFG  2018-01-01                     4.000000                              0.0  
     2018-02-01                     4.000000                              0.0  
     2018-03-01        

In [130]:
print(df.columns.tolist())

['risk_3', 'risk_12', 'logfat_risk_3', 'logfat_risk_12', 'INFORM', 'VU', 'CC', 'HA', 'event_type', 'sub_event_type', 'disorder_type', 'events', 'fatalities', 'hdx_alert_level', 'hdx_value', 'monthly_displacement', 'Flight', 'Airport', 'Travel', 'Train', 'Bus', 'Passport', 'Travel_visa', 'Right_of_asylum', 'rolling_3m_displacements', 'allocation-eligible', 'target_2m', 'risk_gt_06', 'is_protracted', 'disp_6m_avg', 'fat_6m_avg', 'hdx_med_high_count', 'hdx_3m_sum', 'p_sig1', 'p_sig2', 'p_sig3', 'protracted_signal', 'h_sig1', 'h_sig2', 'hard_onset_signal', 'early_signal', 'monthly_displacement_lag1', 'monthly_displacement_lag2', 'fatalities_lag1', 'events_lag1', 'risk_3_lag1', 'risk_12_lag1', 'logfat_risk_3_lag1', 'logfat_risk_12_lag1', 'hdx_alert_max', 'hdx_alert_sum', 'hdx_alert_mean', 'acled_civilian_targeted', 'acled_heavy_warfare', 'acled_social_unrest', 'acled_territorial_shift', 'acled_strategic_or_other', 'acled_political_violence', 'Flight_lag_1m', 'Airport_lag_1m', 'Travel_lag_1m

In [131]:
df.to_csv("../data_clean/final_data.csv")

In [132]:
df_check = df.reset_index() if isinstance(df.index, pd.MultiIndex) else df.copy()
df_check['month'] = pd.to_datetime(df_check['month'])

# Count unique ISO3 and Months
num_countries = df_check['iso3'].nunique()
num_months = df_check['month'].nunique()

print("=" * 50)
print(f"📊 Dataset statistics:")
print(f"   -> Number of unique countries (iso3): {num_countries}")
print(f"   -> Total number of unique months:  {num_months}")
print("=" * 50)

# Check if there are Gaps (missing months) in the global timeline
min_date = df_check['month'].min()
max_date = df_check['month'].max()
expected_range = pd.date_range(start=min_date, end=max_date, freq='MS') # 'MS' = Month Start
missing_months = [m.strftime('%Y-%m-%d') for m in expected_range if m not in df_check['month'].values]

print(f"Temporal analysis (From {min_date.strftime('%Y-%m')} to {max_date.strftime('%Y-%m')}):")
print(f"   -> Expected months: {len(expected_range)}")

if len(missing_months) == 0:
    print("   -> PERFECT: No temporal gaps found in the global dataset. All months are covered.")
else:
    print(f"   -> ALERT: {len(missing_months)} months are missing in the global dataset!")
    print(f"   -> Missing months: {missing_months}")
print("=" * 50)

📊 Dataset statistics:
   -> Number of unique countries (iso3): 176
   -> Total number of unique months:  99
Temporal analysis (From 2018-01 to 2026-03):
   -> Expected months: 99
   -> PERFECT: No temporal gaps found in the global dataset. All months are covered.


### PLOT THE RESULTS:

In [69]:
df_final = pd.read_csv("../data_clean/final_data.csv")

In [74]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def plot_full_crisis_timeline(df, cerf_path, country_code):
    # 1. PREPARAR EL DATAFRAME PRINCIPAL
    if 'iso3' not in df.columns:
        df = df.reset_index()
        
    df_plot = df[df['iso3'] == country_code].copy()
    df_plot['month'] = pd.to_datetime(df_plot['month'])
    df_plot = df_plot.sort_values('month')
    
    if df_plot.empty:
        print(f"No hay datos en el DataFrame principal para el país: {country_code}")
        return

    # Transformación Log(1 + x) para desplazamientos
    df_plot['disp_log1p'] = np.log1p(df_plot['monthly_displacement'].fillna(0))

    # Extraer las fechas de alerta target_2m (marcadas a mitad de mes)
    if 'target_2m' in df_plot.columns:
        warning_months = df_plot[df_plot['target_2m'] == 1]['month']
        warning_dates = warning_months + pd.Timedelta(days=0)
    else:
        warning_dates = pd.Series(dtype='datetime64[ns]')

    # 3. CREAR LA FIGURA INTERACTIVA
    fig = go.Figure()

    # --- B. Desplazamientos (Línea simple morada - Puesta en y1 para que NO desaparezca) ---
    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['disp_log1p'],
        customdata=df_plot['monthly_displacement'], 
        mode='lines', 
        name='Displacements [log(1+x)]',
        line=dict(color='purple', width=2.5),
        yaxis='y1',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Displacements (Real):</b> %{customdata:,.0f}<br><b>log(1+x):</b> %{y:.2f}<extra></extra>'
    ))

    # --- D. Zonas de Conflicto (Fondo sombreado salmón) ---
    if 'allocation-eligible' in df_plot.columns:
        conflict_months = df_plot[df_plot['allocation-eligible'] == 1]['month']
        for c_month in conflict_months:
            fig.add_vrect(
                x0=c_month - pd.Timedelta(days=15), 
                x1=c_month + pd.Timedelta(days=15),
                fillcolor="salmon",
                opacity=0.2,
                layer="below",
                line_width=0,
            )
        # Leyenda del conflicto
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Allocation-eligible Zone',
            line=dict(color='salmon', width=10), opacity=0.3, yaxis='y1'
        ))

    # --- NUEVO: Condición >50k Desplazamientos (Fondo azul clarito) ---
    # Comprueba que tu columna se llame 'target'. Si se llama 'target_4m', cámbialo aquí.
    target_col = 'target' if 'target' in df_plot.columns else None
    if target_col:
        target_months = df_plot[df_plot[target_col] == 1]['month']
        for t_month in target_months:
            fig.add_vrect(
                x0=t_month - pd.Timedelta(days=15), 
                x1=t_month + pd.Timedelta(days=15),
                fillcolor="lightblue",
                opacity=0.3,
                layer="below",
                line_width=0,
            )
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='>50k Displacements (Target)',
            line=dict(color='lightblue', width=10), opacity=0.4, yaxis='y1'
        ))

    # --- F. Líneas conflict_2m (Líneas verticales rojas a rayas) ---
    if not warning_dates.empty:
        for w_date in warning_dates:
            fig.add_shape(
                type="line",
                x0=w_date, x1=w_date,
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(width=2.5, dash="dash", color="red"),
            )
        # Leyenda de Alertas
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Warning (1-2m Pre-Conflict)',
            line=dict(color='red', width=2.5, dash='dash'), yaxis='y1'
        ))

    # --- 4. CONFIGURACIÓN DEL LAYOUT (Eliminados los ejes invisibles) ---
    fig.update_layout(
        title=dict(
            text=f'<b>Full Crisis Timeline Overview: {country_code}</b>',
            font=dict(size=22),
            x=0.05
        ),
        margin=dict(l=60, r=100, t=110, b=60), 
        height=600,
        plot_bgcolor='white',
        hovermode="closest",
        
        xaxis=dict(
            title=dict(text="<b>Date</b>"),
            showgrid=False,
            dtick="M3",
            tickformat="%b %Y",
            tickangle=45
        ),
        
        # Eje Y1: Ahora son los Desplazamientos
        yaxis=dict(
            title=dict(text="<b>log(1 + Displacements)</b>", font=dict(color="purple")),
            tickfont=dict(color="purple"),
            showgrid=True,
            gridcolor='lightgrey',
            rangemode="tozero"
        ),
        
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgrey',
            borderwidth=1
        )
    )

    fig.show()

# --- EJECUCIÓN ---
plot_full_crisis_timeline(df_final, cerf_path="../data_clean/cerf_clean.csv", country_code="SYR")

In [77]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def plot_full_crisis_timeline(df, cerf_path, country_code):
    # 1. PREPARAR EL DATAFRAME PRINCIPAL
    if 'iso3' not in df.columns:
        df = df.reset_index()
        
    df_plot = df[df['iso3'] == country_code].copy()
    df_plot['month'] = pd.to_datetime(df_plot['month'])
    df_plot = df_plot.sort_values('month')
    
    if df_plot.empty:
        print(f"No hay datos en el DataFrame principal para el país: {country_code}")
        return

    # Transformación Log(1 + x) para desplazamientos
    df_plot['disp_log1p'] = np.log1p(df_plot['monthly_displacement'].fillna(0))

    # 2. PREPARAR EL DATAFRAME DE CERF
    try:
        cerf = pd.read_csv(cerf_path)
        cerf = cerf[cerf['iso3'] == country_code].copy()
        cerf['Allocation Date'] = pd.to_datetime(cerf['Allocation Date'], format='ISO8601', errors='coerce')
        cerf = cerf.dropna(subset=['Allocation Date'])
    except FileNotFoundError:
        print(f"No se encontró el archivo CERF en {cerf_path}.")
        cerf = pd.DataFrame()

    # Extraer las fechas de alerta target_2m (marcadas a mitad de mes)
    if 'target_2m' in df_plot.columns:
        warning_months = df_plot[df_plot['target_2m'] == 1]['month']
        warning_dates = warning_months + pd.Timedelta(days=0)
    else:
        warning_dates = pd.Series(dtype='datetime64[ns]')

    # 3. CREAR LA FIGURA INTERACTIVA
    fig = go.Figure()

    # --- A. Fatalidades (Línea simple naranja - y1) ---
    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['fatalities'],
        mode='lines', 
        name='Fatalities',
        line=dict(color='orange', width=2.5),
        yaxis='y1',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Fatalities:</b> %{y:,.0f}<extra></extra>'
    ))

    # --- B. Desplazamientos (Línea simple morada - y2) ---
    fig.add_trace(go.Scatter(
        x=df_plot['month'],
        y=df_plot['disp_log1p'],
        customdata=df_plot['monthly_displacement'], 
        mode='lines', 
        name='Displacements [log(1+x)]',
        line=dict(color='purple', width=2.5),
        yaxis='y2',
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Displacements (Real):</b> %{customdata:,.0f}<br><b>log(1+x):</b> %{y:.2f}<extra></extra>'
    ))

    # --- C. Riesgo (Línea Negra Punteada - y3) ---
    if 'risk_3' in df_plot.columns:
        fig.add_trace(go.Scatter(
            x=df_plot['month'],
            y=df_plot['risk_3'],
            mode='lines',
            name='Risk Score',
            line=dict(color='black', width=2, dash='dash'),
            yaxis='y3',
            hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Risk:</b> %{y:.2f}<extra></extra>'
        ))

    # --- D. Zonas de Conflicto (Fondo sombreado salmón) ---
    if 'allocation-eligible' in df_plot.columns:
        conflict_months = df_plot[df_plot['allocation-eligible'] == 1]['month']
        for c_month in conflict_months:
            fig.add_vrect(
                x0=c_month - pd.Timedelta(days=15), 
                x1=c_month + pd.Timedelta(days=15),
                fillcolor="salmon",
                opacity=0.2,
                layer="below",
                line_width=0,
            )
        # Leyenda del conflicto
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Allocation-eligible Zone',
            line=dict(color='salmon', width=10), opacity=0.3, yaxis='y1'
        ))

    # --- E. Líneas CERF (Líneas verticales verdes punteadas) ---
    if not cerf.empty:
        for _, row in cerf.iterrows():
            fig.add_shape(
                type="line",
                x0=row['Allocation Date'], x1=row['Allocation Date'],
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(width=2.5, dash="dot", color="mediumseagreen"),
            )
        # Leyenda de CERF
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='CERF Allocation',
            line=dict(color='mediumseagreen', width=2.5, dash='dot'), yaxis='y1'
        ))

    # --- F. Líneas conflict_2m (Líneas verticales rojas a rayas) ---
    if not warning_dates.empty:
        for w_date in warning_dates:
            fig.add_shape(
                type="line",
                x0=w_date, x1=w_date,
                y0=0, y1=1,
                xref="x", yref="paper",
                line=dict(width=2.5, dash="dash", color="red"),
            )
        # Leyenda de Alertas
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='lines', name='Warning (1-2m Pre-Conflict)',
            line=dict(color='red', width=2.5, dash='dash'), yaxis='y1'
        ))

   # --- G. HDX Alert Levels ---

    # --- G. HDX Alert Levels ---

    if 'hdx_alert_level' in df_plot.columns:

        hdx_points = []

        for _, row in df_plot.iterrows():

            alerts = str(row['hdx_alert_level']).lower()

            if 'high concern' in alerts:
                hdx_points.append({
                    'month': row['month'],
                    'level': 'High concern',
                    'color': 'red',
                    'y': 1.08
                })

            if 'medium concern' in alerts:
                hdx_points.append({
                    'month': row['month'],
                    'level': 'Medium concern',
                    'color': 'gold',
                    'y': 1.03
                })

        hdx_alerts = pd.DataFrame(hdx_points)

        if not hdx_alerts.empty:

            fig.add_trace(go.Scatter(
                x=hdx_alerts['month'],
                y=hdx_alerts['y'],

                mode='markers',

                name='HDX Alerts',

                marker=dict(
                    size=7,
                    color=hdx_alerts['color'],
                    symbol='circle',
                    line=dict(width=0.5, color='black')
                ),

                yaxis='y3',

                customdata=hdx_alerts['level'],

                hovertemplate=
                    '<b>HDX SIGNAL</b><br>' +
                    'Date: %{x|%b %Y}<br>' +
                    'Level: %{customdata}<extra></extra>'
            ))

    # --- 4. CONFIGURACIÓN DEL LAYOUT MULTI-EJE ---
    fig.update_layout(
        title=dict(
            text=f'<b>Full Crisis Timeline Overview: {country_code}</b>',
            font=dict(size=22),
            x=0.05
        ),
        margin=dict(l=60, r=100, t=110, b=60), 
        height=600,
        plot_bgcolor='white',
        hovermode="closest",
        
        xaxis=dict(
            domain=[0, 0.85], # Recortamos un poco para que quepa el 3r eje a la derecha
            title=dict(text="<b>Date</b>"),
            showgrid=False,
            dtick="M3",
            tickformat="%b %Y",
            tickangle=45
        ),
        
        # Eje Y1: Izquierda (Fatalidades)
        yaxis=dict(
            title=dict(text="<b>Fatalities</b>", font=dict(color="orange")),
            tickfont=dict(color="orange"),
            showgrid=True,
            gridcolor='lightgrey',
            rangemode="tozero"
        ),
        
        # Eje Y2: Derecha Interna (Desplazamientos Log)
        yaxis2=dict(
            title=dict(text="<b>log(1 + Displacements)</b>", font=dict(color="purple")),
            tickfont=dict(color="purple"),
            anchor="x",
            overlaying="y",
            side="right",
            showgrid=False,
            rangemode="tozero"
        ),
        
        # Eje Y3: Derecha Externa (Risk Score)
        yaxis3=dict(
            title=dict(text="<b>Risk</b>", font=dict(color="black")),
            tickfont=dict(color="black"),
            anchor="free",
            overlaying="y",
            side="right",
            position=1.0,
            range=[0, 1.15],
            showgrid=False
        ),
        
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='lightgrey',
            borderwidth=1
        )


    )

    fig.show()

# --- EJECUCIÓN ---
plot_full_crisis_timeline(df, cerf_path="../data_clean/cerf_clean.csv", country_code="SYR")